# 01 Data Understanding
Project: predicting hotel booking cancellations.

Here I look at the raw data before changing anything: size, columns, target and data quality.

Source: Antonio, N., de Almeida, A., & Nunes, L. (2019). *Hotel booking demand datasets.* Data in Brief, 22, 41–49. I use the TidyTuesday copy (`hotels.csv`), which is the same as the Kaggle "Hotel Booking Demand" file.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 40)
df = pd.read_csv("../data/hotel_bookings.csv")
print("Rows × columns:", df.shape)

Rows × columns: (119390, 32)


## 1. First look

In [2]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [4]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
is_canceled,119390.0,0.37,0.48,0.00,0.00,0.00,1.0,1.0
lead_time,119390.0,104.01,106.86,0.00,18.00,69.00,160.0,737.0
arrival_date_year,119390.0,2016.16,0.71,2015.00,2016.00,2016.00,2017.0,2017.0
arrival_date_week_number,119390.0,27.17,13.61,1.00,16.00,28.00,38.0,53.0
arrival_date_day_of_month,119390.0,15.80,8.78,1.00,8.00,16.00,23.0,31.0
stays_in_weekend_nights,119390.0,0.93,1.00,0.00,0.00,1.00,2.0,19.0
stays_in_week_nights,119390.0,2.50,1.91,0.00,1.00,2.00,3.0,50.0
adults,119390.0,1.86,0.58,0.00,2.00,2.00,2.0,55.0
children,119386.0,0.10,0.40,0.00,0.00,0.00,0.0,10.0
babies,119390.0,0.01,0.10,0.00,0.00,0.00,0.0,10.0


## 2. One row
One row is one booking at a City Hotel (Lisbon) or a Resort Hotel (Algarve). Arrivals are from July 2015 to August 2017. There is no booking ID and no personal data.

In [5]:
print(df["hotel"].value_counts())
dates = pd.to_datetime(df["arrival_date_year"].astype(str) + "-" + df["arrival_date_month"] + "-" + df["arrival_date_day_of_month"].astype(str))
print("Arrival dates:", dates.min().date(), "→", dates.max().date())

hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64
Arrival dates: 2015-07-01 → 2017-08-31


## 3. Variable dictionary
Columns grouped by when the hotel knows them. The model can only use what is known at booking time.

In [6]:
dictionary = pd.DataFrame([
    ("hotel", "City Hotel or Resort Hotel", "booking time"),
    ("is_canceled", "TARGET: 1 = booking was canceled", "outcome"),
    ("lead_time", "Days between booking and arrival", "booking time"),
    ("arrival_date_year / month / week_number / day_of_month", "Planned arrival date", "booking time"),
    ("stays_in_weekend_nights / stays_in_week_nights", "Planned nights", "booking time"),
    ("adults / children / babies", "Number of guests", "booking time"),
    ("meal", "Meal package (BB, HB, FB, SC, Undefined)", "booking time"),
    ("country", "Guest country (ISO code)", "booking time*"),
    ("market_segment / distribution_channel", "How the booking was sold (Online TA, Groups, Direct…)", "booking time"),
    ("is_repeated_guest / previous_cancellations / previous_bookings_not_canceled", "Guest history", "booking time"),
    ("reserved_room_type", "Room type booked", "booking time"),
    ("assigned_room_type", "Room type actually given", "AFTER booking (at check-in)"),
    ("booking_changes", "Number of changes until check-in/cancel", "AFTER booking (keeps changing)"),
    ("deposit_type", "No Deposit / Non Refund / Refundable", "booking time"),
    ("agent / company", "ID of travel agent / company that booked", "booking time"),
    ("days_in_waiting_list", "Days before booking was confirmed", "booking time"),
    ("customer_type", "Transient, Contract, Group, Transient-Party", "booking time"),
    ("adr", "Average daily rate (EUR per night)", "booking time"),
    ("required_car_parking_spaces", "Parking spaces requested", "AFTER booking (likely at arrival)"),
    ("total_of_special_requests", "Number of special requests", "booking time"),
    ("reservation_status", "Canceled / Check-Out / No-Show", "OUTCOME (= target)"),
    ("reservation_status_date", "Date of last status", "OUTCOME"),
], columns=["column", "meaning", "known when?"])
dictionary

,column,meaning,known when?
0,hotel,City Hotel or Resort Hotel,booking time
1,is_canceled,TARGET: 1 = booking was canceled,outcome
2,lead_time,Days between booking and arrival,booking time
3,arrival_date_year / month / week_number / day_...,Planned arrival date,booking time
4,stays_in_weekend_nights / stays_in_week_nights,Planned nights,booking time
5,adults / children / babies,Number of guests,booking time
6,meal,"Meal package (BB, HB, FB, SC, Undefined)",booking time
7,country,Guest country (ISO code),booking time*
8,market_segment / distribution_channel,"How the booking was sold (Online TA, Groups, D...",booking time
9,is_repeated_guest / previous_cancellations / p...,Guest history,booking time


## 4. Target

In [7]:
counts = df["is_canceled"].value_counts()
print(counts)
print(f"Cancellation rate: {df['is_canceled'].mean():.1%}")

is_canceled
0    75166
1    44224
Name: count, dtype: int64
Cancellation rate: 37.0%


37% of bookings are canceled. A model that always says "not canceled" already gets 63% accuracy, so I use precision, recall, F1, ROC-AUC and PR-AUC instead.

## 5. Data quality

In [8]:
missing = df.isna().sum()
missing = missing[missing > 0].to_frame("missing")
missing["percent"] = (missing["missing"] / len(df) * 100).round(2)
missing

,missing,percent
children,4,0.00
country,488,0.41
agent,16340,13.69
company,112593,94.31


In [9]:
print("Exact duplicate rows:", df.duplicated().sum(), f"({df.duplicated().mean():.1%})")
print("Bookings with 0 guests:", ((df["adults"] + df["children"].fillna(0) + df["babies"]) == 0).sum())
print("Negative ADR:", (df["adr"] < 0).sum(), "| ADR > 1000:", (df["adr"] > 1000).sum(), "| max ADR:", df["adr"].max())
print("Meal 'Undefined' (same as SC):", (df["meal"] == "Undefined").sum())
print("market_segment 'Undefined':", (df["market_segment"] == "Undefined").sum())

Exact duplicate rows: 31994 (26.8%)
Bookings with 0 guests: 180
Negative ADR: 1 | ADR > 1000: 1 | max ADR: 5400.0
Meal 'Undefined' (same as SC): 1169
market_segment 'Undefined': 2


## 6. Leakage check
Some columns are only filled in after the booking. They make the model look perfect but can't be used at booking time.

In [10]:
pd.crosstab(df["reservation_status"], df["is_canceled"])

is_canceled,0,1
reservation_status,,
Canceled,0,43017
Check-Out,75166,0
No-Show,0,1207


In [11]:
for col in ["required_car_parking_spaces"]:
    print(df.groupby(df[col] > 0)["is_canceled"].mean().rename("cancel rate"))
print(df.groupby(df["reserved_room_type"] != df["assigned_room_type"])["is_canceled"].mean().rename("cancel rate (room changed?)"))

required_car_parking_spaces
False    0.394949
True     0.000000
Name: cancel rate, dtype: float64
False    0.415629
True     0.053764
Name: cancel rate (room changed?), dtype: float64


- `reservation_status` is the target itself (Canceled / No-Show = canceled). Dropped.
- No booking with a parking space was canceled, and bookings with a changed room type cancel only 5%. Both are recorded at check-in, and a canceled guest never checks in. Dropped, together with `booking_changes`, which keeps changing after booking.

## Issues to fix in notebook 02
| Issue | Size | Plan |
|---|---|---|
| `company` missing | 94% | flag `has_company` |
| `agent` missing | 14% | flag `has_agent` |
| `country` missing | 0.4% | fill "Unknown" |
| `children` missing | 4 rows | fill 0 |
| Exact duplicates | 32k rows (27%) | keep (probably real group rooms), check in 04 |
| 0-guest bookings | 180 | drop |
| ADR negative / 5400 | 2 rows | drop |
| Leaky columns | 5 | not used in models |